# Step 2 – Validation of the applicant name harmonisation

This notebook checks the harmonisation built in `step2_harmonization.ipynb`, using the final table of
organisation names exported from it (`org_groups.csv`).


In [10]:
import pandas as pd
from pathlib import Path

# Project folder, found from wherever the kernel starts
ROOT   = next(p for p in [Path.cwd(), *Path.cwd().parents] if p.name == "Group15_innovation")
NB_DIR = ROOT / "step2_validation"           # folder of this notebook and of its input files

# One row per organisation psn_id, with the original name, the cleaned name and the final group
# (exported at the end of step2Ludo_final.ipynb)
org = pd.read_csv(NB_DIR / "org_groups.csv")
org.head()

,psn_id,psn_name,clean,group_final
0,21112,▲復▼旦大学ＦＵＮＤＡＮ ＵＮＩＶＥＲＳＩＴＹ,FUNDAN UNIV,FUNDAN UNIV
1,32932,01 COMMUNIQUE LABORATORY,01 COMMUNIQUE LAB,01 COMMUNIQUE LAB
2,32932,01 COMMUNIQUE LABORATORY,01 COMMUNIQUE LAB,01 COMMUNIQUE LAB
3,32932,01 COMMUNIQUE LABORATORY,01 COMMUNIQUE LAB,01 COMMUNIQUE LAB
4,41455,1QB INFORMATION TECHNOLOGIES,1QB INFORMATION TECH,1QBIT


## 1. Why we do not use the course gold standard

The course gold standard (`PATSTAT_500_label_datav4.csv`, 3,192 names in 481 groups) was built for
another purpose and contains mostly large industrial brands (Siemens, Mitsubishi, Valeo, Toshiba…).
We check (1) how many of our applicants it contains and (2) which names decide the score on it.
Its names are cleaned with a short version of our cleaning, so that the two lists can be compared.

In [11]:
import re, unicodedata

# Course gold standard: search it anywhere under the project folder
found = list(ROOT.rglob("PATSTAT_500_label_datav4.csv"))
if not found:
    raise FileNotFoundError("PATSTAT_500_label_datav4.csv not found under " + str(Path.cwd()) + ": upload it first")
GOLD_FILE = found[0]
print("Gold standard:", GOLD_FILE)
gold = pd.read_csv(GOLD_FILE, encoding="utf-8-sig")     # columns: Group (id of the true group), comp (raw name)

# Short version of the cleaning of the harmonisation notebook, applied to the gold-standard names
LEGAL   = ["CO", "COMPANY", "CORP", "CORPORATION", "INC", "INCORPORATED", "LTD", "LIMITED", "LLC", "LLP",
           "LLLP", "PLC", "GMBH", "AG", "SA", "SAS", "SL", "SPA", "SRL", "BV", "NV", "AB", "OY", "KK", "PTE", "PTY"]
WORDS   = {"UNIVERSITY": "UNIV", "UNIVERSITAET": "UNIV", "UNIVERSITE": "UNIV", "UNIVERSIDAD": "UNIV",
           "UNIVERSITA": "UNIV", "UNIVERSITAT": "UNIV",
           "TECHNOLOGY": "TECH", "TECHNOLOGIES": "TECH", "TECHNOLOGICAL": "TECH",
           "INSTITUTE": "INST", "INSTITUT": "INST", "LABORATORY": "LAB", "LABORATORIES": "LAB",
           "SCIENCES": "SCIENCE", "CENTRE": "CENTER"}
FILLERS = {"THE", "OF", "AND", "UND", "FOR", "DE", "DES", "DU", "DI", "ET", "&"}

def clean_name(x):
    s = unicodedata.normalize("NFKC", str(x)).upper()
    s = "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))   # accents
    s = re.sub(r"[.'’]", "", s)                    # S.A. -> SA, XI'AN -> XIAN
    s = re.sub(r"[^A-Z0-9 ]", " ", s)              # other punctuation and brackets -> space
    words = [WORDS.get(w, w) for w in s.split() if w not in FILLERS]
    while len(words) > 1 and words[-1] in LEGAL:   # legal forms at the end
        words.pop()
    return " ".join(words)

# Check: on our own (Latin-script) names the function reproduces the notebook cleaning
latin = org.drop_duplicates("psn_name")
latin = latin[latin["psn_name"].str.contains("[A-Za-z]")]
print(f"Same cleaned name as the harmonisation notebook: {(latin['psn_name'].map(clean_name) == latin['clean']).mean():.1%}")

# Compare the two lists on cleaned names
gold["clean"] = gold["comp"].map(clean_name)
ours = org.groupby("clean").agg(group_final=("group_final", "first"), links=("psn_id", "size")).reset_index()
gold["in_ours"] = gold["clean"].isin(set(ours["clean"]))
ours["in_gold"] = ours["clean"].isin(set(gold["clean"]))

Gold standard: /home/jovyan/Group15_innovation/entity-linking/PATSTAT_500_label_datav4.csv
Same cleaned name as the harmonisation notebook: 97.8%


In [12]:
# Table 1 - Which of our applicants could the course gold standard validate?
ours["quantum"] = ours["clean"].str.contains("QUANTUM|QUBIT")
ours["univ"]    = ours["clean"].str.contains(r"\bUNIV\b")

rows = {"all our applicants":                      ours,
        "quantum-specific names (QUANTUM, QUBIT)": ours[ours["quantum"]],
        "universities":                            ours[ours["univ"]]}
coverage = pd.DataFrame([{
    "applicants":            name,
    "in our data":           len(df),
    "also in gold standard": int(df["in_gold"].sum()),
    "not in gold standard":  int((~df["in_gold"]).sum()),
    "share validated":       f"{df['in_gold'].mean():.1%}",
} for name, df in rows.items()]).set_index("applicants")
display(coverage)

,in our data,also in gold standard,not in gold standard,share validated
applicants,,,,
all our applicants,2651,65,2586,2.5%
"quantum-specific names (QUANTUM, QUBIT)",184,0,184,0.0%
universities,563,0,563,0.0%


In [13]:
# Table 2 - Which names decide the score on the gold standard?
# Pairwise precision/recall count the pairs of names that the gold standard puts in the same group,
# so a group with n names weighs n*(n-1)/2 pairs.
by_group = (gold.groupby("Group")
                .agg(brand=("comp", lambda s: s.str.split().str[0].mode()[0]),   # most common first word
                     names=("comp", "size"), names_in_ours=("in_ours", "sum"))
                .reset_index())
by_group["pairs"] = by_group["names"] * (by_group["names"] - 1) // 2
by_group = by_group.sort_values("pairs", ascending=False)
by_group["share of the score"] = by_group["pairs"] / by_group["pairs"].sum()
by_group["cumulative share"]   = by_group["share of the score"].cumsum()

top = by_group.head(10)[["brand", "names", "names_in_ours", "share of the score", "cumulative share"]]
display(top.style.format({"share of the score": "{:.1%}", "cumulative share": "{:.1%}"}).hide(axis="index"))

pairs_ours = (by_group["names_in_ours"] * (by_group["names_in_ours"] - 1) // 2).sum()



brand,names,names_in_ours,share of the score,cumulative share
SIEMENS,198,5,21.2%,21.2%
MITSUBISHI,179,5,17.3%,38.5%
PHOENIX,119,1,7.6%,46.2%
VALEO,111,0,6.6%,52.8%
TOSHIBA,104,8,5.8%,58.6%
BASF,87,0,4.1%,62.7%
SUMITOMO,84,2,3.8%,66.5%
SONY,78,2,3.3%,69.8%
PHILIPS,76,1,3.1%,72.9%
SCHLUMBERGER,64,6,2.2%,75.1%


In [14]:
# Table 3 - Our names that fall in the 10 groups deciding 75% of the gold-standard score
top10 = by_group.head(10)[["Group", "brand"]]

# Gold names of these groups that are also in our data, matched to our cleaned names
in_top = (gold[gold["in_ours"]]
            .merge(top10, on="Group")
            .drop_duplicates("clean")                      # one row per cleaned name of ours
            .merge(ours[["clean", "group_final", "links"]], on="clean"))

share = in_top["links"].sum() / ours["links"].sum()
print(f"{len(in_top)} of our cleaned names, with {in_top['links'].sum()} of "
      f"{ours['links'].sum():,} family-applicant links ({share:.1%} of our data)")

21 of our cleaned names, with 229 of 17,751 family-applicant links (1.3% of our data)


**Results.**

*Table 1.* Had we used the gold standard, only **65 of our 2,651 applicants (2.5%)** could have been checked,
because the other names are simply not in it. None of the 184 quantum-specific applicants and none of the
563 universities are in the gold standard: the actors that matter most for a quantum landscape would not
be validated at all.

*Table 2.* Pairwise precision and recall count the pairs of names that the gold standard puts in the same
group, so large groups dominate the score: the **ten largest groups (Siemens, Mitsubishi, Phoenix, Valeo,
Toshiba, BASF…) decide 75% of it**, and only 0.13% of all pairs involve two of our names. A high score on
the gold standard therefore mostly means grouping correctly the subsidiaries of large industrial brands,
most of which are outside quantum computing.

*Table 3.* Of these ten heavy groups, only **21 of our cleaned names** are in our data, with **229 of 17,751
family-applicant links (1.3%)**. Most of them are large incumbents with few quantum families (Siemens, Sony,
Philips, Schlumberger, Sumitomo); the only relevant quantum actor is Toshiba, which is already one of our
128 hand-checked anchors. For some of these names the gold standard would also judge us by a different rule:
it puts Mitsubishi Electric and Mitsubishi Chemical in the same group (one brand, separate companies, which we
keep apart because they have no common majority owner) and places "Phoenix Company of Chicago" in a group of
unrelated firms that share the word "Phoenix".

For these reasons we validate the harmonisation on a random sample of our own names (section 2).

## 2. Validation on a sample of our names

The 128 anchors and all their candidate names were already checked by hand (three independent annotators),
so checking them again would only confirm our own labels. We therefore **exclude every name in the candidates
file** (as anchor or as candidate) and sample from the remaining names, i.e. the part of the data harmonised
only by the automatic steps (cleaning + typo merges).

In [16]:
from pathlib import Path

# Inputs
CANDIDATES_FILE = ROOT / "outputs" / "step2" / "candidates_top128.csv"
# Output folder for this notebook
OUT_DIR = ROOT / "outputs" / "step2" / "validation"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_FILE = OUT_DIR / "validation_sample.csv"          # file to label by hand

N_SAMPLE = 100
SEED     = 42                                            # fixed seed: same sample at every run

# Names already reviewed by hand: anchors and candidates
cand = pd.read_csv(CANDIDATES_FILE)
reviewed = set(cand["anchor"].str.strip()) | set(cand["candidate"].str.strip())
print("Names already reviewed by hand:", len(reviewed))

Names already reviewed by hand: 450


In [17]:
# Population: organisation names never seen in the hand review.
# We check both the cleaned name and the original name, because the candidates file
# contains cleaned names (plus a few original CJK names).
already_seen = (org["clean"].astype(str).str.strip().isin(reviewed)
                | org["psn_name"].astype(str).str.strip().isin(reviewed))
pool = org.loc[~already_seen, ["psn_name", "group_final"]].drop_duplicates("psn_name")

print("Organisation names in total:", org["psn_name"].nunique())
print("Excluded (already reviewed): ", org.loc[already_seen, "psn_name"].nunique())
print("Validation population:       ", len(pool))

Organisation names in total: 3279
Excluded (already reviewed):  651
Validation population:        2628


**Sampling.** We draw a simple random sample of 100 names, with a fixed seed so that the sample
(and our labels) stay the same every time the notebook is run.

A simple random sample mostly contains small applicants (most names have one or two families), so it
measures the quality of the automatic steps on the tail. The head is covered by the hand review above.
Each name is judged against the whole dataset, not only against the other 99 names: a pairwise
precision/recall inside 100 random names would be almost empty, because two random tail names
almost never belong to the same organisation.

In [18]:
import os

# Write the file to label only once, so that a re-run does not overwrite our labels
if os.path.exists(SAMPLE_FILE):
    print(SAMPLE_FILE, "already exists: not overwritten")
else:
    sample = pool.sample(n=N_SAMPLE, random_state=SEED).sort_values("psn_name")
    sample["error"] = ""          # to fill by hand: 0 = correct, 1 = wrong
    sample.to_csv(SAMPLE_FILE, index=False, encoding="utf-8-sig")
    print(SAMPLE_FILE, "written:", len(sample), "names")

/home/jovyan/Group15_innovation/outputs/step2/validation/validation_sample.csv already exists: not overwritten


**Labelling rule** (column `error`):

- **0** – `group_final` is the right organisation for this name.
- **1** – the name is in the wrong group (it was merged with a different organisation), **or** it should belong
  to another existing group but was left on its own (e.g. a subsidiary or a spelling variant of a larger group).

The rule for "same organisation" is the one used for the harmonisation: majority owner at filing date
(subsidiaries, IP companies and name changes included; joint ventures and spin-offs kept apart).

In [19]:
# Read the labelled sample and compute name-level accuracy, precision and recall
lab = pd.read_csv(OUT_DIR / "validation_sample_filled.csv", encoding="utf-8-sig")
lab["error"] = pd.to_numeric(lab["error"])
lab["error_type"] = lab["error_type"].fillna("")
n = len(lab)

def wilson(k, n, z=1.96):
    """95% Wilson interval for a proportion k/n."""
    p = k / n
    centre = (p + z**2 / (2*n)) / (1 + z**2 / n)
    half   = z * ((p*(1-p)/n + z**2 / (4*n**2)) ** 0.5) / (1 + z**2 / n)
    return centre - half, centre + half

# Counts of the two kinds of error
wrong_merges  = (lab["error_type"] == "wrong_merge").sum()    # name grouped with a different organisation
missed_merges = (lab["error_type"] == "missed_merge").sum()   # name left out of its organisation's group
correct       = (lab["error"] == 0).sum()

# Name-level measures: share of names without each kind of error
results = pd.DataFrame({
    "measure": ["accuracy", "precision", "recall"],
    "meaning": ["name in the right group",
                "group not mixed with other organisations",
                "name not left out of its organisation's group"],
    "ok":      [correct, n - wrong_merges, n - missed_merges],
})
results["value"] = results["ok"] / n
ci = [wilson(k, n) for k in results["ok"]]
results["ci_low"]  = [lo for lo, hi in ci]
results["ci_high"] = [hi for lo, hi in ci]
p, r = results.loc[1, "value"], results.loc[2, "value"]
f1 = 2*p*r / (p + r)

print(f"Sample: {n} names, {n - correct} errors "
      f"({wrong_merges} wrong merges, {missed_merges} missed merges)")
display(results.round(3))
print(f"F1 (precision-recall): {f1:.3f}")


Sample: 100 names, 17 errors (0 wrong merges, 17 missed merges)


,measure,meaning,ok,value,ci_low,ci_high
0,accuracy,name in the right group,83,0.83,0.745,0.891
1,precision,group not mixed with other organisations,100,1.00,0.963,1.000
2,recall,name not left out of its organisation's group,83,0.83,0.745,0.891


F1 (precision-recall): 0.907


**Results.** This check concerns only the **tail** of the data. The names reviewed by hand (the 128 anchors
with ≥ 20 families and all their candidates) are excluded on purpose: they were already checked by three
independent annotators, and they are also the most important names, because they cover 58% of the
family–applicant links and include all the largest applicants. The 100 names checked here were harmonised
only by the automatic steps (cleaning + typo merges), and most of them have one or two families.

On these 100 names, 83 are in the right group (accuracy 0.83, 95% CI 0.75–0.89).
All 17 errors are **missed merges**; there is **no wrong merge** (precision 1.00, 95% CI 0.96–1.00).
With no wrong merges, accuracy and recall coincide.

This is what we wanted from a conservative method: in the worst case it leaves an organisation split
into two or more groups, but it never puts two different organisations together.
